# 第十三課 - 使用 Cognee 知識圖譜的代理記憶


## 設定

此筆記本示範如何使用 [**Cognee**](https://www.cognee.ai/) 知識圖譜和 **Microsoft Agent Framework** (MAF) 建立具備持久記憶的智能 <strong>編碼助理</strong>。

Cognee 將非結構化文本轉換為結構化且可查詢的知識圖譜，並以向量嵌入支撐 —— 為你的代理賦予豐富且關係感知的長期記憶。

### 你將學習到的內容
1. <strong>建立知識圖譜</strong>：將開發者簡介和最佳實踐轉換成結構化且可查詢的知識。
2. **將 Cognee 與 MAF 整合**：使用 `@tool` 函數讓 MAF 代理查詢 Cognee 的知識圖譜。
3. <strong>具會話感知的對話</strong>：在同一會話中維持多個問題的上下文。
4. <strong>長期記憶</strong>：在多次會話間持久保存重要知識，並在新對話中檢索。

### 預備條件
- Python 3.9+
- 本機執行 Redis (`docker run -d -p 6379:6379 redis`) 作會話管理
- 一組 LLM API 金鑰 (例如 OpenAI) — 在 `.env` 設定 `LLM_API_KEY`
- `.env` 設定 `CACHING=true`（Cognee 會話所需）
- 具已部署聊天模型的 Microsoft Foundry 專案
- `.env` 裡設定 `AZURE_AI_PROJECT_ENDPOINT` 和 `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI 已登入認證 (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## 代理記憶類型

本筆記本探討與主課程第13課筆記本相同的三種記憶類型，但使用 Cognee 作為長期記憶後端：

| 記憶類型 | 機制 | 壽命 |
|---|---|---|
| <strong>工作中</strong> | `agent.create_session()` (MAF) | 單次對話 |
| <strong>短期</strong> | Cognee 會話快取 (Redis) | 單次會話 |
| <strong>長期</strong> | Cognee 知識圖譜 + 向量 | 永久 |

### Cognee 的記憶架構
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## 準備 Cognee 儲存


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## 第1部分 — 建立知識庫

我們攝取三種類型的數據來為我們的編碼助理建立一個全面的知識庫：

1. <strong>開發者檔案</strong> — 個人專業知識和技術背景
2. **Python最佳實踐** — Python之禪與實用指導方針
3. <strong>歷史對話</strong> — 過去開發者與AI助理之間的問答環節


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## 視覺化知識圖譜

Cognee 可以呈現它所擷取的實體和關係的互動式 HTML 視覺化。


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## 使用 Memify 豐富記憶

`memify()` 會分析知識圖譜並生成智能規則 —— 識別模式、最佳實踐及概念之間的關係。


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## 第二部分 — 具有 Cognee 工具的 MAF 代理

現在我們建立一個 MAF 代理，該代理可以通過 `@tool` 函數查詢 Cognee 的知識圖。這讓代理能夠利用圖譜感知語義搜索的全部威力，同時通過會話保持對話上下文。


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## 有會話的工作記憶

`AgentSession`（透過 `agent.create_session()` 建立）於會話期間提供工作記憶。代理人可以回顧之前的訊息，同時查詢 Cognee 的長期知識圖譜。


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## 新會話 — 長期記憶持續存在

開始一個新的會話會清除工作記憶，但 Cognee 知識圖譜仍然可用。代理人可以在全新的對話中檢索相同的長期知識。


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## 摘要

在這個筆記本中，您建構了一個結合 **MAF 的工作記憶**（`agent.create_session()`）與 **Cognee 的長期知識圖譜** 的編碼助手。

### 您學到了什麼
1. <strong>知識圖譜建構</strong>：Cognee 會攝取非結構化文字並建構圖譜 + 向量記憶。
2. **利用 memify 強化圖譜**：在現有圖譜之上衍生事實與更豐富的關係。
3. **MAF + Cognee 整合**：`@tool` 函數讓 MAF 代理能自然查詢 Cognee 的圖譜。
4. **工作記憶 + 長期記憶**：`AgentSession`（透過 `agent.create_session()`）提供會話上下文，而 Cognee 則提供持久的知識。
5. **利用 NodeSets 篩選查找**：針對知識圖譜中的特定子集（例如僅限原則）進行查找。

### 主要重點
- **Cognee** 將原始文字轉換為結構化的、具關係意識的記憶 — 比單純向量庫更強大。
- **`@tool` 函數** 為 MAF 代理和外部知識系統搭建了清晰的橋樑。
- **`AgentSession`**（透過 `agent.create_session()`）將每次對話的上下文與長期知識分開保存。
- 同一個知識圖譜可服務多個會話和代理。

### 實際應用
- <strong>開發者輔助工具</strong>：程式碼審查、事件分析、架構助理
- <strong>面向客戶的輔助工具</strong>：基於產品文件、FAQ 和 CRM 記錄的客戶支援代理
- <strong>內部專家輔助工具</strong>：根據政策、法律或安全指南推理的助手
- <strong>統一數據層</strong>：將結構化與非結構化數據合併成一個可查詢圖譜

### 接下來的步驟
- 在 Cognee 中實驗時間感知能力
- 為特定領域圖譜品質定義 OWL 本體
- 新增用戶反饋迴路，隨時間提升檢索能力
- 擴展到多代理系統共享同一 Cognee 記憶層


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
